# S4 — Shadow projection

Stage 4 of the Manhattan Sidewalk Shade Index pipeline.

For each hourly timestep (08:00–18:00), computes solar position via `pvlib`, projects every tree crown as an offset shadow circle (vectorised — no per-tree Python loop), and overlays the per-timestep shadow union against every analysis unit. Per `docs/DECISIONS.md`: crown-offset trig and solar position stay in Python (`numpy`/`pvlib`); the union + overlay against analysis units runs in **DuckDB** (`ST_Union_Agg` + `ST_Intersection`), which handles all ~62k trees × 11 timesteps well inside a couple of seconds per timestep — no rasterised fallback needed.

**Accept when:** shadows point away from the sun (morning→west, afternoon→east), shadow area grows toward the ends of the day, no empty union at midday.

In [1]:
from pathlib import Path
from datetime import datetime

import yaml
import numpy as np
import pandas as pd
import geopandas as gpd
import duckdb
import pvlib

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
config = yaml.safe_load(open(PROJECT_ROOT / "config.yaml"))
ANALYSIS_CRS = config["analysis_crs"]
INTERIM = PROJECT_ROOT / config["interim_dir"]
print("s4: Project shadows")
print(f"Timestamp: {datetime.now().isoformat()}\n")

s4: Project shadows
Timestamp: 2026-08-28T14:09:19.236839



## Load trees-with-crowns and analysis units; compute solar positions

One solar position per hour for the Manhattan centroid (documented simplification, per CLAUDE.md §4 — azimuth/altitude vary <1° across the borough at a given hour).

In [2]:
trees = gpd.read_parquet(PROJECT_ROOT / config["output"]["units_with_crowns"])
units = gpd.read_parquet(PROJECT_ROOT / config["output"]["analysis_units"])
print(f"trees: {len(trees):,}  units: {len(units):,}")

MANHATTAN_CENTROID_LATLON = (40.7831, -73.9712)
times = pd.date_range(
    start=f"{config['reference_date']} {config['start_hour']}:00",
    end=f"{config['reference_date']} {config['end_hour']}:00",
    freq=f"{config['step_hours']}h",
    tz=config["timezone"],
)
solpos = pvlib.solarposition.get_solarposition(times, *MANHATTAN_CENTROID_LATLON)
print(solpos[["azimuth", "apparent_elevation"]])

trees: 62,416  units: 34,603
                              azimuth  apparent_elevation
2025-07-15 08:00:00-04:00   82.149160           24.580338
2025-07-15 09:00:00-04:00   91.625150           35.889938
2025-07-15 10:00:00-04:00  102.718652           47.134330
2025-07-15 11:00:00-04:00  117.668304           57.785926
2025-07-15 12:00:00-04:00  141.252141           66.610125
2025-07-15 13:00:00-04:00  178.642441           70.618360
2025-07-15 14:00:00-04:00  216.744879           67.046960
2025-07-15 15:00:00-04:00  241.105165           58.412243
2025-07-15 16:00:00-04:00  256.419898           47.822886
2025-07-15 17:00:00-04:00  267.669649           36.590300
2025-07-15 18:00:00-04:00  277.203125           25.265790


## Per-timestep: vectorised shadow circles, DuckDB union + overlay

For each tree, the shadow circle has the *same radius as the crown*, centred at the trunk offset by `d = crown_centre_height_m / tan(altitude)` in the direction `azimuth + 180°` (opposite the sun). Offsetting the trunk point first and buffering the shifted points is fully vectorised — `GeoSeries.buffer()` accepts a per-row distance array, so no per-tree Python loop is needed for ~62k trees.

Circles are grouped by nearby analysis unit and unioned *before* intersecting (`ST_Union_Agg` then `ST_Intersection`) so overlapping crowns over the same unit don't get double-counted — this is why the union has to happen before the overlay, not after.

In [3]:
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

units_scratch = INTERIM / "_scratch_units_s4.parquet"
units[["unit_id", "geometry"]].to_parquet(units_scratch)
con.execute(f"CREATE TABLE units AS SELECT unit_id, geometry AS geom FROM read_parquet('{units_scratch.as_posix()}')")

trunk_x = trees.geometry.x.to_numpy()
trunk_y = trees.geometry.y.to_numpy()
crown_centre_height = trees["crown_centre_height_m"].to_numpy()
crown_radius = trees["crown_radius_m"].to_numpy()

results = []
min_altitude = config["min_altitude_deg"]

for ts, row in solpos.iterrows():
    altitude, azimuth = row["apparent_elevation"], row["azimuth"]
    label = ts.strftime("%H%M")
    if altitude <= min_altitude:
        print(f"{label}: altitude={altitude:.1f} <= {min_altitude} -- skipped (grazing sun)")
        continue

    alt_rad = np.radians(altitude)
    az_rad = np.radians(azimuth + 180)  # opposite the sun
    d = crown_centre_height / np.tan(alt_rad)
    dx = d * np.sin(az_rad)
    dy = d * np.cos(az_rad)

    shifted = gpd.points_from_xy(trunk_x + dx, trunk_y + dy)
    shadow_circles = gpd.GeoSeries(shifted, crs=ANALYSIS_CRS).buffer(crown_radius)
    circles_gdf = gpd.GeoDataFrame({"geometry": shadow_circles}, crs=ANALYSIS_CRS)
    circles_scratch = INTERIM / f"_scratch_circles_{label}.parquet"
    circles_gdf.to_parquet(circles_scratch)

    con.execute(f"CREATE OR REPLACE TABLE circles AS SELECT geometry AS geom FROM read_parquet('{circles_scratch.as_posix()}')")
    shaded = con.execute("""
        SELECT u.unit_id, ST_Area(ST_Intersection(u.geom, ST_Union_Agg(c.geom))) AS shaded_area_m2
        FROM units u JOIN circles c ON ST_Intersects(u.geom, c.geom)
        GROUP BY u.unit_id, u.geom
    """).fetchdf()
    circles_scratch.unlink()

    total_shadow_area = shaded["shaded_area_m2"].sum()
    print(f"{label}: az={azimuth:6.1f} alt={altitude:5.1f}  "
          f"shaded units={len(shaded):,}  total shaded area={total_shadow_area:,.0f} m2  "
          f"mean offset dx={dx.mean():.1f} dy={dy.mean():.1f}")

    out_path = INTERIM / f"shadows_{label}.parquet"
    shaded.to_parquet(out_path)
    results.append({"label": label, "azimuth": azimuth, "altitude": altitude,
                     "n_shaded_units": len(shaded), "total_shadow_area_m2": total_shadow_area,
                     "mean_dx": dx.mean(), "mean_dy": dy.mean()})

results_df = pd.DataFrame(results)
units_scratch.unlink()

0800: az=  82.1 alt= 24.6  shaded units=8,549  total shaded area=237,626 m2  mean offset dx=-13.3 dy=-1.8


0900: az=  91.6 alt= 35.9  shaded units=9,971  total shaded area=189,094 m2  mean offset dx=-8.5 dy=0.2


1000: az= 102.7 alt= 47.1  shaded units=15,852  total shaded area=417,494 m2  mean offset dx=-5.6 dy=1.3


1100: az= 117.7 alt= 57.8  shaded units=17,473  total shaded area=612,874 m2  mean offset dx=-3.4 dy=1.8


1200: az= 141.3 alt= 66.6  shaded units=17,970  total shaded area=715,770 m2  mean offset dx=-1.7 dy=2.1


...

1400: az= 216.7 alt= 67.0  shaded units=17,863  total shaded area=647,654 m2  mean offset dx=1.6 dy=2.1


1500: az= 241.1 alt= 58.4  shaded units=17,521  total shaded area=488,824 m2  mean offset dx=3.3 dy=1.8


1600: az= 256.4 alt= 47.8  shaded units=15,611  total shaded area=319,506 m2  mean offset dx=5.4 dy=1.3


1700: az= 267.7 alt= 36.6  shaded units=13,525  total shaded area=240,144 m2  mean offset dx=8.3 dy=0.3


1800: az= 277.2 alt= 25.3  shaded units=12,002  total shaded area=218,946 m2  mean offset dx=12.9 dy=-1.6


## QA summary and direction spot-check

In [4]:
print("\n" + "=" * 70)
print("S4 — Shadow projection: QA Summary")
print("=" * 70)
print(results_df.to_string(index=False))
print("=" * 70)

assert len(results_df) > 0, "no valid timesteps -- every hour was below min_altitude_deg"
assert (results_df["n_shaded_units"] > 0).all(), "a timestep had zero shaded units"

morning = results_df[results_df["label"].astype(int) <= 1000]
afternoon = results_df[results_df["label"].astype(int) >= 1600]
if len(morning):
    assert (morning["mean_dx"] < 0).all(), "morning shadows should shift west (negative x)"
    print(f"Morning (<=10:00) shadows shift west: OK (mean dx {morning['mean_dx'].mean():.1f} m)")
if len(afternoon):
    assert (afternoon["mean_dx"] > 0).all(), "afternoon shadows should shift east (positive x)"
    print(f"Afternoon (>=16:00) shadows shift east: OK (mean dx {afternoon['mean_dx'].mean():.1f} m)")

midday = results_df[(results_df["label"].astype(int) >= 1100) & (results_df["label"].astype(int) <= 1300)]
assert (midday["total_shadow_area_m2"] > 0).all(), "midday shadow union must not be empty"

print("\nAll S4 acceptance checks passed.")
print(f"\nWrote {len(results_df)} shadows_HHMM.parquet files to {INTERIM}")
print("\ns4 complete. Ready for S5 (shade index).")


S4 — Shadow projection: QA Summary
label    azimuth  altitude  n_shaded_units  total_shadow_area_m2    mean_dx   mean_dy
 0800  82.149160 24.580338            8549         237625.930482 -13.312152 -1.835573
 0900  91.625150 35.889938            9971         189093.691406  -8.491289  0.240913
 1000 102.718652 47.134330           15852         417494.108484  -5.565175  1.256070
 1100 117.668304 57.785926           17473         612873.661657  -3.430124  1.798435
 1200 141.252141 66.610125           17970         715769.583810  -1.664064  2.073539
 1300 178.642441 70.618360           17989         722316.449810  -0.051232  2.161833
 1400 216.744879 67.046960           17863         647654.329532   1.557400  2.086001
 1500 241.105165 58.412243           17521         488823.682044   3.309218  1.826397
 1600 256.419898 47.822886           15611         319506.351392   5.413468  1.307666
 1700 267.669649 36.590300           13525         240144.145961   8.272846  0.336661
 1800 277.203125 2